# Operations Research 3: Examples for Theory

This code is for students who take the class Operations Research. Students should finish the installation of Gurobi and Python before workersed and make sure an academic liscense for Gurobi is applied and activated.

We introduce an example for Network flow problem in order to help students understand how to implement theories introduced in lectures with codes. More instruction is provided in the lecture video.

# Network Flow Problem
## Example: Assignment Problem

Import instance of an assignment problem. \
In the file, we have the arcs of a bipartite graph. Each arc means if the job $i$ is assigned to worker $j$, we should pay the cost $c_{ij}$. \
Each worker and job should be assigned no greater than one time in an assignment problem.

In [2]:
import gurobipy as gp
import pandas as pd
import numpy as np

In [3]:
graph_info = pd.read_excel('bipartite_arcs_set.xlsx', 'graph')
print(graph_info)

   jobs workers  cost
0     A       E     2
1     A       F  9999
2     A       G    10
3     A       H     7
4     B       E  9999
5     B       F     4
6     B       G     3
7     B       H  9999
8     C       E     8
9     C       F     2
10    C       G     5
11    C       H  9999
12    D       E     7
13    D       F  9999
14    D       G     1
15    D       H     6


In [10]:
graph_info.shape

(16, 3)

In [11]:
graph_info.head()

,jobs,workers,cost
0,A,E,2
1,A,F,9999
2,A,G,10
3,A,H,7
4,B,E,9999


In [12]:
graph_info.columns

Index(['jobs', 'workers', 'cost'], dtype='str')

In [13]:
graph_info.info()

<class 'pandas.DataFrame'>
RangeIndex: 16 entries, 0 to 15
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   jobs     16 non-null     str  
 1   workers  16 non-null     str  
 2   cost     16 non-null     int64
dtypes: int64(1), str(2)
memory usage: 516.0 bytes


Prepare data into different format to make us construct model more efficiently.

In [15]:
jobs = ["A", "B", "C", "D"]  # set I
works = ["E", "F", "G", "H"]  # set J
jobs_dict = {}
works_dict = {}
for i in range(len(jobs)):
    jobs_dict[jobs[i]] = i
for i in range(len(works)):
    works_dict[works[i]] = i
print(jobs_dict)
print(works_dict)

{'A': 0, 'B': 1, 'C': 2, 'D': 3}
{'E': 0, 'F': 1, 'G': 2, 'H': 3}


In [25]:
# Incidence Matrix and Cost Matrix
inci_matrix = np.zeros((len(jobs), len(works)))
cost_matrix = np.zeros((len(jobs), len(works)))
for i in range(len(graph_info)):
    inci_matrix[
        jobs_dict[graph_info['jobs'][i]],
        works_dict[graph_info['workers'][i]]
    ] = 1
print(inci_matrix)
for i in range(len(graph_info)):
    cost_matrix[
        jobs_dict[graph_info['jobs'][i]],
        works_dict[graph_info['workers'][i]]
    ] = graph_info['cost'][i]
print(cost_matrix)

[[1. 1. 1. 1.]
 [1. 1. 1. 1.]
 [1. 1. 1. 1.]
 [1. 1. 1. 1.]]
[[2.000e+00 9.999e+03 1.000e+01 7.000e+00]
 [9.999e+03 4.000e+00 3.000e+00 9.999e+03]
 [8.000e+00 2.000e+00 5.000e+00 9.999e+03]
 [7.000e+00 9.999e+03 1.000e+00 6.000e+00]]


After preparing data, we construct a model to solve the assignment problem.


In [35]:
eg3 = gp.Model("Assignment")

# Add variables
x = np.empty((len(jobs), len(works)), dtype=object)
for i in range(len(jobs)):
    for j in range(len(works)):
        x[i, j] = eg3.addVar(lb=0, ub=1, vtype=gp.GRB.CONTINUOUS, name=f"x{(i, j)}")

# Set Objective Value
eg3.setObjective(np.sum(cost_matrix * x), gp.GRB.MINIMIZE)

# Constraints: Each job and worker must be assigned
# For the variable matrix, each row and each column sum to 1, respectively
for i in range(x.shape[0]):
    eg3.addConstr(gp.quicksum(x[i, j] for j in range(x.shape[1])) == 1, name=f"worker {i}")
for i in range(x.shape[1]):
    eg3.addConstr(gp.quicksum(x[j, i] for j in range(x.shape[0])) == 1, name=f"job {i}")

# Call the solver
eg3.optimize()

# Print results
if eg3.status == gp.GRB.OPTIMAL: # gp.GRB.OPTIMAL is a named constant: 2 optimal, 3 infeasible, 5 unbounded
    for var in eg3.getVars():
        print(var.varName, "=", var.x)
print(f"Objective value = {eg3.objVal}")


Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 9 5950X 16-Core Processor, instruction set [SSE2|AVX|AVX2]
Thread count: 16 physical cores, 32 logical processors, using up to 32 threads

Optimize a model with 8 rows, 16 columns and 32 nonzeros (Min)
Model fingerprint: 0x2e4a36ab
Model has 16 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]
Presolve time: 0.00s
Presolved: 8 rows, 16 columns, 32 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    8.0000000e+00   2.000000e+00   0.000000e+00      0s
       2    1.3000000e+01   0.000000e+00   0.000000e+00      0s

Solved in 2 iterations and 0.01 seconds (0.00 work units)
Optimal objective  1.300000000e+01
x(0, 0) = 1.0
x(0, 1) = 0.0
x(0, 2) = 0.0
x(0, 3) = 0.0
x(1, 0) = 0.0
x(1, 1) = 0.0
x(1, 2) = 1.0
x(1, 3) = 0